In [ ]:
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
import numpy as np
import pickle
import json
from pathlib import Path

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
ctl = 'goamazon_2pulse.largedom.r20251008.rerun'
ehe1 = 'goamazon_2pulse.largedom.ehe1.r20251030.rerun'
with open(f'{ctl}/pkl/csd_stats.sparse.claude.pkl', 'rb') as f:
    stat_ctl = pickle.load(f)
with open(f'{ehe1}/pkl/csd_stats.sparse.claude.pkl', 'rb') as f:
    stat_ehe1 = pickle.load(f)

In [ ]:
nc_ehe1, = stat_ehe1[0].shape
nc_ctl, = stat_ctl[0].shape
print(f'{nc_ehe1} clouds in EHE1 simulation')
print(f'{nc_ctl} clouds in CTL simulation')

In [ ]:
dz = 50. # m
z = np.arange(dz/2, 5000., dz)
dts = 30 # seconds

In [ ]:
minmf = 0
print(np.log10(np.max(stat_ehe1[0])))
print(np.log10(np.max(stat_ctl[0])))
maxmf = np.max([np.max(stat_ehe1[0]), np.max(stat_ctl[0])]) + 10.0
# maxmf = np.max(stat_ehe1[0]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)
mf_bins = np.linspace(minmf, maxmf, 11)

In [ ]:
time_bins = np.arange(0, 241, 30)
print(time_bins)

In [ ]:
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
n, time_bins, patches = ax.hist([stat_ehe1[-4], stat_ctl[-4]], bins=time_bins, log=True, color=['red', 'black'])
ax.legend(['EHE1', 'CTL'])
ax.set_ylim(1, 3.0e4)
ax.set_xlabel('cloud ini time')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i in range(len(time_bins) - 1):
    ax = axes[i]
    
    # Filter clouds by initialization time bin
    mask_ehe1 = (stat_ehe1[-4] > time_bins[i]) & (stat_ehe1[-4] <= time_bins[i+1])
    mask_ctl = (stat_ctl[-4] > time_bins[i]) & (stat_ctl[-4] <= time_bins[i+1])
    
    # Plot histograms for clouds in this time bin
    n_bin, _, _ = ax.hist([np.log10(stat_ehe1[0][mask_ehe1]), np.log10(stat_ctl[0][mask_ctl])], 
                          bins=mf_bins, log=True, color=['red', 'black'], alpha=0.7)
    
    ax.set_xlabel(f'Mean cloud-base mass flux (kg/s)')
    ax.set_ylabel('Count')
    ax.set_title(f'Time bin: {time_bins[i]:.0f}-{time_bins[i+1]:.0f}')
    ax.legend(['EHE1', 'CTL'])
    ax.set_ylim(0.5, 1.0e4)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate difference histograms for each time bin
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i in range(len(time_bins) - 1):
    ax = axes[i]
    
    # Filter clouds by initialization time bin
    mask_ehe1 = (stat_ehe1[-4] > time_bins[i]) & (stat_ehe1[-4] <= time_bins[i+1])
    mask_ctl = (stat_ctl[-4] > time_bins[i]) & (stat_ctl[-4] <= time_bins[i+1])
    
    # Calculate histograms for this time bin
    n_ehe1, _ = np.histogram(np.log10(stat_ehe1[0][mask_ehe1]), bins=mf_bins)
    n_ctl, _ = np.histogram(np.log10(stat_ctl[0][mask_ctl]), bins=mf_bins)
    
    # Calculate difference
    diff = n_ehe1 - n_ctl
    bin_centers = (mf_bins[:-1] + mf_bins[1:]) / 2
    
    # Plot difference bars
    ax.bar(bin_centers, np.where(diff > 0, diff, np.nan), 
           width=np.diff(mf_bins), color='red', alpha=0.7, label=r'EHE1 $>$ CTL')
    ax.bar(bin_centers, np.where(diff < 0, -diff, np.nan), 
           width=np.diff(mf_bins), color='blue', alpha=0.7, label=r'CTL $>$ EHE1')
     
    ax.axhline(0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Mean cloud-base mass flux (log10 kg/s)')
    ax.set_ylabel('Absolute Difference in Count')
    ax.set_yscale('log')
    ax.set_ylim([0.1, 1.0e3])
    ax.set_title(f'Time bin: {time_bins[i]:.0f}-{time_bins[i+1]:.0f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
n, mf_bins, patches = ax.hist([np.log10(stat_ehe1[0]), np.log10(stat_ctl[0])], bins=mf_bins, log=True, color=['red', 'black'])
ax.axvline(np.median(np.log10(stat_ehe1[0])), color='red', linestyle='--')
ax.axvline(np.median(np.log10(stat_ctl[0])), color='black', linestyle='--')
ax.legend(['EHE1', 'CTL'])
ax.set_ylim(1, 3.0e4)
ax.set_xlabel(f'Mean cloud-base mass flux (kg/s)')
plt.show()

In [ ]:
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
diff = n[0] - n[1]
print(diff)
bin_centers = (mf_bins[:-1] + mf_bins[1:]) / 2
ax.bar(bin_centers, np.where(diff>0.0, diff, np.nan), width=np.diff(mf_bins), color='red', label=r'EHE1 $>$ CTL')
ax.bar(bin_centers, np.where(diff>0.0, np.nan, -diff), width=np.diff(mf_bins), color='blue', label=r'CTL $>$ EHE1')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Mean cloud-base mass flux (kg/s)')
ax.set_ylabel('Difference (EHE1 - CTL)')
ax.set_yscale('log')
ax.set_ylim(0.1, 1.0e4)
ax.set_title('Difference between EHE1 and CTL histograms')
ax.legend()
plt.show()

In [ ]:
def compare_csd_norm(stat_ehe, stat_ctl, name_ehe, nbins, minmf, maxmf):
    bins = np.linspace(minmf, maxmf, nbins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_widths = np.diff(bins)

    # Use only positive, finite values before log10
    ehe_raw = np.asarray(stat_ehe[0], dtype=float)
    ctl_raw = np.asarray(stat_ctl[0], dtype=float)
    ehe_log = np.log10(ehe_raw[(ehe_raw > 0) & np.isfinite(ehe_raw)])
    ctl_log = np.log10(ctl_raw[(ctl_raw > 0) & np.isfinite(ctl_raw)])

    # Normalized histogram (fraction per bin)
    h_ehe, _ = np.histogram(ehe_log, bins=bins)
    h_ctl, _ = np.histogram(ctl_log, bins=bins)

    h_ehe = h_ehe / h_ehe.sum()
    h_ctl = h_ctl / h_ctl.sum()

    # Convert to %
    h_ehe_pct = 100.0 * h_ehe
    h_ctl_pct = 100.0 * h_ctl
    diff_pct = h_ehe_pct - h_ctl_pct
    print(diff_pct)

    fig = plt.figure(figsize=(12, 14))
    ax1 = fig.add_axes((0.1, 0.55, 0.85, 0.4))
    ax2 = fig.add_axes((0.1, 0.1, 0.85, 0.4))

    # Panel 1: normalized distributions
    ax1.bar(bin_centers, h_ctl_pct, width=bin_widths, color='black', alpha=0.45, label='CTL')
    ax1.bar(bin_centers, h_ehe_pct, width=bin_widths, color='red', alpha=0.45, label=name_ehe)
    ax1.axvline(np.median(ehe_log), color='red', linestyle='--')
    ax1.axvline(np.median(ctl_log), color='black', linestyle='--')
    ax1.set_yscale('log')
    ax1.set_ylabel('Cloud fraction per bin (%)')
    ax1.set_title('Normalized histograms')
    ax1.legend()

    ax2.bar(bin_centers, np.where(diff_pct>0.0, diff_pct, np.nan), width=bin_widths, color='red', label=rf'{name_ehe} $>$ CTL')
    ax2.bar(bin_centers, np.where(diff_pct>0.0, np.nan, -diff_pct), width=bin_widths, color='blue', label=rf'CTL $>$ {name_ehe}')
    ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel(r'$\log_{10}\left<M_b\right>$')
    ax2.set_ylabel(f'Difference ({name_ehe} - CTL)')
    ax2.set_yscale('log')
    ax2.set_ylim(1.0e-2, 20.)
    ax2.legend()

    plt.show()

In [ ]:
def compare_csd_norm2(stat_ehe, stat_ctl, name_ehe, nbins, minmf, maxmf):
    bins = np.linspace(minmf, maxmf, nbins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_widths = np.diff(bins)

    # Use only positive, finite values before log10
    ehe_raw = np.asarray(stat_ehe[0], dtype=float)
    ctl_raw = np.asarray(stat_ctl[0], dtype=float)
    ehe_log = np.log10(ehe_raw[(ehe_raw > 0) & np.isfinite(ehe_raw)])
    ctl_log = np.log10(ctl_raw[(ctl_raw > 0) & np.isfinite(ctl_raw)])

    # Normalized histogram (fraction per bin)
    h_ehe, _ = np.histogram(ehe_log, bins=bins)
    h_ctl, _ = np.histogram(ctl_log, bins=bins)

    h_ehe = h_ehe / h_ehe.sum()
    h_ctl = h_ctl / h_ctl.sum()

    # Convert to %
    h_ehe_pct = 100.0 * h_ehe
    h_ctl_pct = 100.0 * h_ctl
    diff_pct = 100.0*(h_ehe_pct - h_ctl_pct)/h_ctl_pct
    print(diff_pct)

    fig = plt.figure(figsize=(12, 14))
    ax1 = fig.add_axes((0.1, 0.55, 0.85, 0.4))
    ax2 = fig.add_axes((0.1, 0.1, 0.85, 0.4))

    # Panel 1: normalized distributions
    ax1.bar(bin_centers, h_ctl_pct, width=bin_widths, color='black', alpha=0.45, label='CTL')
    ax1.bar(bin_centers, h_ehe_pct, width=bin_widths, color='red', alpha=0.45, label=name_ehe)
    ax1.axvline(np.median(ehe_log), color='red', linestyle='--')
    ax1.axvline(np.median(ctl_log), color='black', linestyle='--')
    ax1.set_yscale('log')
    ax1.set_ylabel('Cloud fraction per bin (%)')
    ax1.set_title('Normalized histograms')
    ax1.legend()

    ax2.bar(bin_centers, np.where(diff_pct>0.0, diff_pct, np.nan), width=bin_widths, color='red', label=rf'{name_ehe} $>$ CTL')
    ax2.bar(bin_centers, np.where(diff_pct>0.0, np.nan, -diff_pct), width=bin_widths, color='blue', label=rf'CTL $>$ {name_ehe}')
    ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel(r'$\log_{10}\left<M_b\right>$')
    ax2.set_ylabel(f'Difference ({name_ehe} - CTL)')
    ax2.set_yscale('log')
    ax2.set_ylim(1.0e-2, 100.)
    ax2.legend()

    plt.show()

In [ ]:
compare_csd_norm(stat_ehe1, stat_ctl, "EHE1", 10, minmf, maxmf)

In [ ]:
compare_csd_norm(stat_ehe1, stat_ctl, "EHE1", 15, minmf, maxmf)

In [ ]:
compare_csd_norm2(stat_ehe1, stat_ctl, "EHE1", 15, minmf, maxmf)